In [24]:
import pandas as pd 
import numpy as np    
import matplotlib.pyplot as plt        
import matplotlib.gridspec as gridspec     
import matplotlib.patches as  mpatches
import matplotlib.ticker as mticker 
from matplotlib.lines import Line2D       
from scipy import stats                  
from scipy.stats import skew, kurtosis, norm, t as t_dist             
from scipy.stats import gaussian_kde, probplot

import matplotlib 
matplotlib.use('Agg')

BG_DARK   = '#0D1117'
BG_PANEL  = '#161B22'
BORDER    = '#30363D'
TEXT_PRI  = '#E6EDF3'
TEXT_SEC  = '#8B949E'
GRID_CLR  = '#21262D'
 
COLORS = {
    'MASI (Maroc)'   : '#F0A500',
    'EGX30 (Egypte)' : '#2DD4BF',
    'JSE (Afr.Sud)' : '#3B82F6',
    'NGSE (Nigéria)' : '#F43F5E',
   
}
CLR_POS = '#2DD4BF'
CLR_NEG = '#F43F5E'
 
matplotlib.rcParams.update({
    'figure.facecolor': BG_DARK, 'axes.facecolor': BG_PANEL,
    'axes.edgecolor': BORDER, 'axes.labelcolor': TEXT_PRI,
    'xtick.color': TEXT_SEC, 'ytick.color': TEXT_SEC,
    'text.color': TEXT_PRI, 'grid.color': GRID_CLR,
    'grid.linewidth': 0.5, 'font.family': 'DejaVu Sans', 'font.size': 8,
})
 
CRISES = [
    {'name':'GFC 2008',        'start':'2008-09-01','end':'2009-03-31','color':'#F43F5E'},
    {'name':'Arab Spring',     'start':'2011-01-01','end':'2012-06-30','color':'#FB923C'},
    {'name':'Oil Crash',       'start':'2014-07-01','end':'2016-02-29','color':'#FBBF24'},
    {'name':'COVID-19',        'start':'2020-01-01','end':'2021-06-30','color':'#A78BFA'},
    {'name':'Russie-Ukraine',  'start':'2022-02-01','end':'2023-03-31','color':'#38BDF8'},
    {'name':'Iran-Israël/Gaza','start':'2023-10-01','end':'2025-04-30','color':'#34D399'},
    {'name':'Iran-USA-2026',   'start':'2026-02-01','end':'2026-04-01','color':'#EF4444'}
]




# chargement et nettoyage des données : 
def parse_investing_csv(path, market_name) :
    df=pd.read_csv(path)
    # nettoyage de colonne date :
    df['Date']=pd.to_datetime(df['Date'],  format='%d/%m/%Y')
    df=df.sort_values('Date').reset_index(drop=True)
    
    
    # nettoyage de la colonne de variation %
    var_col='Variation %'
    df[var_col]=(
        df[var_col].astype(str)
        .str.replace('%','',regex=False)
        .str.replace(',','.', regex=False)
        .str.strip()
    )
    
    df[var_col]=pd.to_numeric(df[var_col], errors='coerce')/100
    
    
    def clean_price(s):
        s=str(s).strip()
        s=s.replace('.','').replace(',','.')
        try:
            return float(s)
        except:
            return np.nan
        
    df['Prix']=df['Dernier'].apply(clean_price)
    
    # mettre la date l'index de dataframe
    df=df.set_index('Date')
    df.index.name='Date'
    
    
    returns=df[var_col].rename(market_name)
    prices=df['Prix'].rename(market_name)
    
    
    # compter les nombres des observations valides
    n_valid=returns.dropna().shape[0]
    
    
    print(f" {market_name:20s} : {n_valid} observations valides : De {df.index[0]} à  {df.index[-1]}")
        
    return returns, prices
   
    


print("chargement des données ")
ret_masi,px_masi=parse_investing_csv('Moroccan All Shares - Données Historiques.csv', 'MASI (Maroc)')

ret_egx,px_egx=parse_investing_csv('EGX 30 - Données Historiques.csv','EGX30 (Egypte)')

ret_jse,px_jse=parse_investing_csv('FTSE_JSE All Share - Données Historiques.csv','JSE (Afr.Sud)')

ret_ngse, px_ngse=parse_investing_csv('NSE All Share - Données Historiques.csv','NGSE (Nigéria)')



# asssemblage dans une Data Frame 
returns_df=pd.concat([ret_masi, ret_egx, ret_jse,ret_ngse], axis=1)

# alignement uniquement les dates où on a au moins de 3 marchés
returns_df=returns_df[returns_df[['MASI (Maroc)','EGX30 (Egypte)','JSE (Afr.Sud)']].notna().all(axis=1)]
markets=list(returns_df.columns)
masi_col = 'MASI (Maroc)'
masi_ret = returns_df[masi_col].dropna()





def align_price(px_series, dates): 
    px=px_series.reindex(dates)
    px=px.ffill()
    base=px.dropna().iloc[0]
    return (px/base)*100


price_idx=pd.DataFrame(index=returns_df.index)

price_idx['MASI (Maroc)']=align_price(px_masi, returns_df.index)
price_idx['EGX30 (Egypte)']=align_price(px_egx, returns_df.index)
price_idx['JSE (Afr.Sud)']=align_price(px_jse, returns_df.index)
price_idx['NGSE (Nigéria)']=align_price(px_ngse, returns_df.index)



# le test wald GMMM de normalité des rendements

def wald_normality_test(r):
    
    # préparation des données
    r=np.asarray(r.dropna())
    n=len(r)
    mu=r.mean()
    v=r.var(ddof=1)
    # calcul des moments 3 et 4
    sk=skew(r)
    xku=kurtosis(r,fisher=True)
    
    # calcul des résidus
    e3=(r-mu)**3/v**1.5-sk
    e4=(r-mu)**4/v*2-3-xku
    # Matrice de covariance variance
    M=np.column_stack([e3,e4])
    S=(M.T@M)/n
    g=np.array([sk,xku])
    try :
        W=n*g@np.linalg.inv(S)@g
    except:
        W=n*(sk**2/6+xku**2/24)
    
    return W,1-stats.chi2.cdf(W,df=2),sk,xku


stats_rows={}
for col in markets :
    r=returns_df[col].dropna()
    mu=r.mean()
    sg=r.std(ddof=1)
    W,pv,sk,xku=wald_normality_test(r)
    sharpe=(mu*12)/(sg*np.sqrt(12)) if sg>0 else 0
    VaR95=norm.ppf(0.05,mu,sg)
    tail_r=r[r<r.quantile(0.05)]
    es95=tail_r.mean() if len(tail_r)>0 else np.nan
    
    stats_rows[col] = {
        'rendement_mensuel': round(mu,4),
        'rendement_annuel': round(mu*12,4),
        'volatilité_mensuel': round(sg,4),
        'volatilité_annuelle': round(sg*np.sqrt(12),4),
        'Autocorrélation': round(r.autocorr(1),4),
        'skewness': round(sk,4),
        'Kurtosis':round(xku,4),
        'Wald khi_deux [GMM]': round(W,3),
        'p_value': round(pv,4),
        'sharpe_ratio': round(sharpe,4),
        'N observations':int(len(r)),
        'Min mensuel observé': round(r.min(),4),
        'Max mensuel obseré': round(r.max(),4),
        '95% CI r1': round(1.96/np.sqrt(len(r)),4),
        'VaR 95% (Normal)': round(VaR95,4),
        'ES95% (hist)':round(es95,4)
        
    }

stats_table=pd.DataFrame(stats_rows)
print("Table 1 : Statistiques descriptives")
print(stats_table.to_string())


# Elements visuels précalculés:

# KDE kernel density function MASI
kde_x=np.linspace(masi_ret.min()*1.6, masi_ret.max()*1.6,500) # création de l'axe X

   
kde_y=gaussian_kde(masi_ret)(kde_x) # création de KDE des données (réalité )

# création de la courbe normale théorique 
norm_y=norm.pdf(kde_x,masi_ret.mean(), ret_masi.std())


# QQcplot 
r_std=(masi_ret-masi_ret.mean())/(masi_ret.std())
qq=probplot(r_std,dist="norm")
qq_th=qq[0][0]
qq_em=qq[0][1]
tail_m=np.abs(qq_th)>1.8

# Radar chart
def norm01(v, inv=False):
    v=np.array(v, dtype=float)
    mn,mx=v.min(),v.max()
    if mx-mn< 1e-12: return np.ones_like(v)*0.5
    n=0.1+0.9*(v-mn)/(mx-mn)
    return 1.1-n if inv else n
rho1v=[abs(stats_table.loc['Autocorrélation',c]) for c in markets]
xkuv=[max(0,stats_table.loc['Kurtosis',c]) for c in markets]
sigv=[stats_table.loc['volatilité_annuelle',c] for c in markets]
varv=[abs(stats_table.loc['VaR 95% (Normal)',c]) for c in markets]
sharpev=[max(0,stats_table.loc['sharpe_ratio', c]) for c in markets]

radar_labels=['r1\n (liquidité)','xku\n (fat tail)','volatilité annuelle','VaR \n (asymètrie)', 'sharpe \n (Performance)']

radar_data= {
    m:[norm01(rho1v, inv=True)[i], norm01(xkuv)[i],norm01(sigv, inv=True)[i],norm01(varv, inv=True)[i], norm01(sharpev)[i]]
    for i ,m in enumerate(markets)
}

colors_list = [COLORS[m] for m in markets]
    


# Construction du dashborad

fig = plt.figure(figsize=(22, 30), facecolor=BG_DARK, dpi=150)

# titre principal 

fig.text(0.5, 0.9885,
         'ANATOMY OF VOLATILITY IN AFRICAN FRONTIER MARKETS',
         ha='center', fontsize=17, fontweight='bold', color=TEXT_PRI)
fig.text(0.5, 0.9840,
         'An Application of the Bekaert-Harvey (1995) Framework to the Casablanca Stock Exchange and Peer African Indices',
         ha='center', fontsize=8.5, color=TEXT_SEC)

sep = fig.add_axes([0.03, 0.980, 0.94, 0.0006])
sep.set_facecolor('#F0A500'); sep.axis('off')

# GridSpec :

gs = gridspec.GridSpec(5, 3,
    top=0.974, bottom=0.025,
    left=0.04, right=0.97,
    hspace=0.52, wspace=0.34)

ax11 = fig.add_subplot(gs[0, :])
ax12 = fig.add_subplot(gs[1, :])
ax13 = fig.add_subplot(gs[2, 0])
ax14 = fig.add_subplot(gs[2, 1])
ax15 = fig.add_subplot(gs[2, 2])
ax16 = fig.add_subplot(gs[3, 0])
ax17 = fig.add_subplot(gs[3, 1], polar=True)
ax18 = fig.add_subplot(gs[3, 2])
ax_T = fig.add_subplot(gs[4, :])

def hdr(ax, num, title, sub=''):
    ax.text(0.0, 1.065, f'Figure {num} — {title}',
            transform=ax.transAxes, fontsize=9, fontweight='bold',
            color=TEXT_PRI, ha='left', va='bottom')
    if sub:
        ax.text(1.0, 1.065, sub, transform=ax.transAxes,
                fontsize=6.5, color=TEXT_SEC, ha='right', va='bottom')

# FIGURE 1.1 : Rendements mensuels MASI 

ax = ax11; ax.set_facecolor(BG_PANEL)

colors_bars = np.where(masi_ret.values >= 0, CLR_POS, CLR_NEG)
for date, ret, c in zip(masi_ret.index, masi_ret.values, colors_bars):
    alpha = 0.88 if abs(ret) > 0.03 else 0.70
    ax.bar(date, ret*100, width=22, color=c, alpha=alpha, linewidth=0)

mu_pct = masi_ret.mean()*100
ax.axhline(0, color=TEXT_SEC, lw=0.7, ls='--', alpha=0.5)
ax.axhline(mu_pct, color=COLORS[masi_col], lw=1.3, ls='-', alpha=0.85,
           label=f'μ = {mu_pct:.3f}%/mois')

for cr in CRISES:
    cs = pd.Timestamp(cr['start']); ce = pd.Timestamp(cr['end'])
    cs = max(cs, masi_ret.index[0]); ce = min(ce, masi_ret.index[-1])
    if cs < ce:
        ax.axvspan(cs, ce, alpha=0.13, color=cr['color'], zorder=1)
        mid = cs + (ce - cs)/2
        ax.text(mid, 0.945, cr['name'],
                transform=ax.get_xaxis_transform(),
                ha='center', va='top', fontsize=6.5, fontweight='bold',
                color=cr['color'],
                bbox=dict(boxstyle='round,pad=0.2', fc=BG_DARK,
                          ec=cr['color'], lw=0.6, alpha=0.88))

ax.legend(loc='upper left', fontsize=7.5, facecolor=BG_PANEL,
          edgecolor=BORDER, labelcolor=TEXT_PRI, framealpha=0.92)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:.1f}%'))
ax.set_xlim(masi_ret.index[0], masi_ret.index[-1])
margin = max(abs(masi_ret.values))*100*0.1
ax.set_ylim(masi_ret.min()*100 - margin, masi_ret.max()*100 + margin*1.5)
ax.grid(axis='y', alpha=0.35, lw=0.4); ax.set_axisbelow(True)
hdr(ax,'1.1','Rendements Mensuels du MASI (2004-2026)',
    f'Période : Jan 2004 - Avr 2026 / Fréquence : Mensuelle / n = {len(masi_ret)} obs ')
ax.text(0.99,-0.07,'Source : Données historiques réelles — MASI, EGX30, JSE All Share, NGX All Share (Investing.com)',
        transform=ax.transAxes, fontsize=5.5, color=TEXT_SEC, ha='right')




# FIGURE 1.2 : Rendements cumulés comparés 

ax = ax12; ax.set_facecolor(BG_PANEL)

for col in price_idx.columns:
    c = COLORS[col]; lw = 2.4 if col==masi_col else 1.5
    label = col 
    ax.plot(price_idx.index, price_idx[col],
            color=c, lw=lw, alpha=0.9, label=label, zorder=3)

ax.fill_between(price_idx.index, 100, price_idx[masi_col],
                color=COLORS[masi_col], alpha=0.07)
ax.axhline(100, color=TEXT_SEC, lw=0.6, ls=':', alpha=0.5)

for cr in CRISES:
    cs = pd.Timestamp(cr['start'])
    if cs >= price_idx.index[0]:
        ax.axvline(cs, color=cr['color'], lw=0.6, ls='--', alpha=0.4)

leg = ax.legend(loc='upper left', fontsize=7.5, facecolor=BG_PANEL,
                edgecolor=BORDER, labelcolor=TEXT_PRI, framealpha=0.92,
                ncol=5, columnspacing=0.8)
ax.set_xlim(price_idx.index[0], price_idx.index[-1])
ax.set_ylim(0, price_idx.max().max()*1.10)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{int(x)}'))
ax.grid(axis='y', alpha=0.35, lw=0.4); ax.set_axisbelow(True)
ax.set_ylabel('Indice cumulé (base 100)', fontsize=7, color=TEXT_SEC)
ax.text(0.99,-0.07,' Données historiques réelles — Investing.com (MASI, EGX30, JSE, NGSE)',
        transform=ax.transAxes, fontsize=5.5, color=TEXT_SEC, ha='right')
hdr(ax,'1.2','Rendements Cumulés Comparés : 4 Indices Africains (base 100, Jan 2004)')

# FIGURE 1.3 : KDE vs Normale 

ax = ax13; ax.set_facecolor(BG_PANEL)

ax.fill_between(kde_x*100, kde_y/100, alpha=0.38, color=COLORS[masi_col])
ax.plot(kde_x*100, kde_y/100, color=COLORS[masi_col], lw=2,
        label='MASI (empirique)')
ax.plot(kde_x*100, norm_y/100, color=TEXT_SEC, lw=1.4,
        ls='--', alpha=0.8, label='Normale théorique')

xku_m = stats_table.loc['Kurtosis', masi_col]
ax.text(0.97,0.94, f'xku = {xku_m:.2f}',
        transform=ax.transAxes, ha='right', va='top',
        fontsize=8.5, fontweight='bold', color=COLORS[masi_col],
        bbox=dict(boxstyle='round,pad=0.3',fc=BG_DARK,
                  ec=COLORS[masi_col], lw=0.8, alpha=0.92))

thr = 1.96*masi_ret.std()
for mask in [kde_x < (masi_ret.mean()-thr), kde_x > (masi_ret.mean()+thr)]:
    if mask.any():
        diff = np.maximum(0, kde_y[mask] - norm_y[mask])
        ax.fill_between(kde_x[mask]*100, 0, diff/100, alpha=0.35, color='#F43F5E')

ax.legend(fontsize=6.5, loc='upper left', facecolor=BG_PANEL,
          edgecolor=BORDER, labelcolor=TEXT_PRI)
ax.set_xlabel('Rendement mensuel (%)', fontsize=7, color=TEXT_SEC)
ax.set_ylabel('Densité', fontsize=7, color=TEXT_SEC)
ax.set_xlim(kde_x.min()*100, kde_x.max()*100)
ax.set_ylim(bottom=0); ax.grid(alpha=0.3)
hdr(ax,'1.3','KDE vs Normale (MASI)')

# FIGURE 1.4 : Q-Q Plot

ax = ax14; ax.set_facecolor(BG_PANEL)

x_line = np.linspace(qq_th.min(), qq_th.max(), 100)
ax.plot(x_line, x_line, color=TEXT_SEC, lw=1, ls='--', alpha=0.7)
ax.scatter(qq_th[~tail_m], qq_em[~tail_m],
           color=COLORS[masi_col], s=18, alpha=0.7, lw=0, zorder=3)
ax.scatter(qq_th[tail_m], qq_em[tail_m],
           color='#F43F5E', s=28, alpha=0.9, lw=0.4,
           edgecolors='white', label='Queue (déviation)', zorder=4)

ax.text(0.05,0.90,'Déviation aux extrémités\n→ preuve de fat tails',
        transform=ax.transAxes, fontsize=6.5, color='#F43F5E',
        bbox=dict(boxstyle='round,pad=0.3',fc=BG_DARK,
                  ec='#F43F5E', lw=0.6, alpha=0.88))
ax.legend(fontsize=6.5, facecolor=BG_PANEL, edgecolor=BORDER, labelcolor=TEXT_PRI)
ax.set_xlabel('Quantiles théoriques N(0,1)', fontsize=7, color=TEXT_SEC)
ax.set_ylabel('Quantiles empiriques MASI', fontsize=7, color=TEXT_SEC)
ax.grid(alpha=0.3)
hdr(ax,'1.4','Q-Q Plot (MASI vs Normale)')

# FIGURE 1.5 : Autocorrélation ρ₁ 

ax = ax15; ax.set_facecolor(BG_PANEL)

rho1_vals = [stats_table.loc['Autocorrélation', m] for m in markets]
ci_avg    = np.mean([stats_table.loc['95% CI r1', m] for m in markets])

bars = ax.bar(range(len(markets)), rho1_vals,
              color=colors_list, alpha=0.85, width=0.6, lw=0)
ax.axhline(ci_avg, color=TEXT_SEC, lw=0.9, ls='--', alpha=0.6,
           label=f'95% CI (≈±{ci_avg:.3f})')
ax.axhline(-ci_avg, color=TEXT_SEC, lw=0.9, ls='--', alpha=0.6)
ax.axhline(0, color=BORDER, lw=0.5)

for bar, val in zip(bars, rho1_vals):
    off = 0.004 if val >= 0 else -0.007
    va  = 'bottom' if val >= 0 else 'top'
    ax.text(bar.get_x()+bar.get_width()/2, val+off,
            f'{val:.3f}', ha='center', va=va,
            fontsize=7, fontweight='bold', color=TEXT_PRI)

short_names = [m.split('(')[0].strip()+'\n('+m.split('(')[1] for m in markets]
ax.set_xticks(range(len(markets))); ax.set_xticklabels(short_names, fontsize=6)
ax.legend(fontsize=6.5, facecolor=BG_PANEL, edgecolor=BORDER, labelcolor=TEXT_PRI)
ax.set_ylabel('Autocorrélation ρ₁', fontsize=7, color=TEXT_SEC)
ax.grid(axis='y', alpha=0.3); ax.set_axisbelow(True)
hdr(ax,'1.5','ρ₁ par Marché')

# FIGURE 1.6 : Kurtosis en excès 

ax = ax16; ax.set_facecolor(BG_PANEL)

xku_vals = [stats_table.loc['Kurtosis', m] for m in markets]
bars = ax.bar(range(len(markets)), xku_vals,
              color=colors_list, alpha=0.85, width=0.6, lw=0)
ax.axhline(0, color=TEXT_SEC, lw=1.0, ls='--', alpha=0.7,
           label='Normale (xku=0)')

for bar, val in zip(bars, xku_vals):
    off = 0.04 if val >= 0 else -0.07
    ax.text(bar.get_x()+bar.get_width()/2, val+off,
            f'{val:.2f}', ha='center',
            va='bottom' if val >= 0 else 'top',
            fontsize=8, fontweight='bold', color=TEXT_PRI)

ax.set_xticks(range(len(markets))); ax.set_xticklabels(short_names, fontsize=6)
ax.legend(fontsize=6.5, facecolor=BG_PANEL, edgecolor=BORDER, labelcolor=TEXT_PRI)
ax.set_ylabel('Kurtosis en excès (xku)', fontsize=7, color=TEXT_SEC)
ax.grid(axis='y', alpha=0.3); ax.set_axisbelow(True)
hdr(ax,'1.6','Kurtosis en Excès par Marché')

# FIGURE 1.7 : Radar Chart 

ax = ax17; ax.set_facecolor(BG_PANEL)
N = len(radar_labels)
angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
angles += angles[:1]

for r_grid in [0.25, 0.50, 0.75, 1.0]:
    circ = np.linspace(0, 2*np.pi, 150)
    ax.plot(circ, [r_grid]*150, color=BORDER, lw=0.5, alpha=0.7)

for ang in angles[:-1]:
    ax.plot([ang, ang], [0, 1.05], color=BORDER, lw=0.5, alpha=0.6)

for mname, vals in radar_data.items():
    vv = vals + [vals[0]]
    c  = COLORS[mname]
    lw = 2.5 if mname == masi_col else 1.3
    ax.plot(angles, vv, color=c, lw=lw, alpha=0.9)
    ax.fill(angles, vv, color=c,
            alpha=0.15 if mname == masi_col else 0.07)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_labels, fontsize=7, color=TEXT_SEC)
ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(['','','',''], fontsize=0)
ax.set_ylim(0, 1.1); ax.grid(False)
hdr(ax17, '1.7', 'Profil de Risque\nMultidimensionnel')
ax.legend()

# FIGURE 1.8 : Volatilité annualisée 

ax = ax18; ax.set_facecolor(BG_PANEL)

vol_vals = [stats_table.loc['volatilité_annuelle', m]*100
            for m in markets]
bars = ax.barh(range(len(markets)), vol_vals,
               color=colors_list, alpha=0.85, height=0.55, lw=0)
ax.axvline(15.0, color=TEXT_SEC, lw=1.1, ls='--', alpha=0.7,
           label='S&P 500 ≈ 15%')

for bar, val in zip(bars, vol_vals):
    ax.text(val+0.3, bar.get_y()+bar.get_height()/2,
            f'{val:.1f}%', ha='left', va='center',
            fontsize=7.5, fontweight='bold', color=TEXT_PRI)

ax.set_yticks(range(len(markets)))
ax.set_yticklabels(markets, fontsize=7, color=TEXT_SEC)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:.0f}%'))
ax.legend(fontsize=6.5, loc='lower right', facecolor=BG_PANEL,
          edgecolor=BORDER, labelcolor=TEXT_PRI)
ax.set_xlabel('σ annualisée )', fontsize=7, color=TEXT_SEC)
ax.grid(axis='x', alpha=0.3); ax.set_axisbelow(True)
ax.set_xlim(0, max(vol_vals)*1.28)
hdr(ax,'1.8','Volatilité Annualisée')

# sTABLE 1

ax = ax_T; ax.set_facecolor(BG_PANEL); ax.axis('off')

rows_show = [
    'rendement_mensuel', 'rendement_annuel',
    'volatilité_mensuel', 'volatilité_annuelle',
    'Autocorrélation', 'skewness',
    'Kurtosis', 'Wald khi_deux [GMM]',
    'p_value', 'sharpe_ratio',
    'N observations', 'Min mensuel observé', 'Max mensuel obseré',
]

def fmt(v, row):
    if pd.isna(v): return '—'
    if 'N obs' in row: return f'{int(v)}'
    if any(k in row for k in ['Rendement','Volatil','Min','Max']):
        return f'{v*100:.3f}%'
    if 'p-value' in row: return f'{v:.4f}'
    if 'χ²' in row or 'Wald' in row: return f'{v:.3f}'
    return f'{v:.4f}'

cell_text = [
    [fmt(stats_table.loc[r, c], r) for c in markets]
    for r in rows_show
]

# Marqueur données réelles vs simulées


tbl = ax.table(
    cellText=cell_text, rowLabels=rows_show,
    colLabels=markets, cellLoc='center',
    rowLoc='left', loc='center',
    bbox=[0.0, 0.0, 1.0, 1.0]
)
tbl.auto_set_font_size(False); tbl.set_fontsize(7.2)

highlight_rows = {'Kurtosis', 'Autocorrélation',
                  'Wald khi_deux [GMM]', 'skewness'}

for (row, col), cell in tbl.get_celld().items():
    cell.set_edgecolor(BORDER); cell.set_linewidth(0.4)
    if row == 0:
        if col >= 0:
            mk = markets[col]
            cell.set_facecolor(COLORS[mk])
            cell.set_text_props(color='#0D1117', fontweight='bold', fontsize=7.2)
        else:
            cell.set_facecolor('#0D1117')
            cell.set_text_props(color=TEXT_SEC, fontweight='bold')
    elif col == -1:
        cell.set_facecolor('#0D1117')
        rn = rows_show[row-1]
        fc = COLORS[masi_col] if rn in highlight_rows else TEXT_PRI
        fw = 'bold' if rn in highlight_rows else 'normal'
        cell.set_text_props(color=fc, fontsize=6.8, fontweight=fw)
    else:
        cell.set_facecolor('#0F1923' if row%2==0 else BG_PANEL)
        fc_cell = COLORS[masi_col] if (col==0 and row>0) else TEXT_PRI
        fw_cell = 'bold' if col==0 else 'normal'
        cell.set_text_props(color=fc_cell, fontsize=7.2, fontweight=fw_cell)

ax.text(0.5, 1.025,
        "Table 1 — Statistiques Descriptives Comparées (style Bekaert-Harvey, 1995)",
        transform=ax.transAxes, ha='center', va='bottom',
        fontsize=9, fontweight='bold', color=TEXT_PRI)





plt.savefig(
    'ANATOMY OF VOLATILITY IN AFRICAN FRONTIER MARKETS.png',
    dpi=150,
    bbox_inches='tight',
    facecolor=fig.get_facecolor(),
    edgecolor='none'
)







 
 

                     
 
    





chargement des données 
 MASI (Maroc)         : 268 observations valides : De 2004-01-01 00:00:00 à  2026-04-01 00:00:00
 EGX30 (Egypte)       : 267 observations valides : De 2004-01-01 00:00:00 à  2026-04-01 00:00:00
 JSE (Afr.Sud)        : 268 observations valides : De 2004-01-01 00:00:00 à  2026-04-01 00:00:00
 NGSE (Nigéria)       : 267 observations valides : De 2004-02-01 00:00:00 à  2026-04-01 00:00:00
Table 1 : Statistiques descriptives
                     MASI (Maroc)  EGX30 (Egypte)  JSE (Afr.Sud)  NGSE (Nigéria)
rendement_mensuel          0.0065          0.0178         0.0099          0.0110
rendement_annuel           0.0777          0.2132         0.1191          0.1314
volatilité_mensuel         0.0434          0.0882         0.0436          0.0727
volatilité_annuelle        0.1502          0.3054         0.1510          0.2517
Autocorrélation            0.1149          0.1544        -0.0564          0.1509
skewness                  -0.1191          0.1827        -0.1666  

C:\Users\monassib.ma\AppData\Local\Temp\ipykernel_13320\3791323396.py:496: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax.legend()
